In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


EXPR_PATH = r"/Cell_line_RMA_proc_basalExp.txt"
CELL_LINES_PATH = r"/Cell_Lines_Details.xlsx"
DOSE_PATH = r"/GDSC2_fitted_dose_response_27Oct23.xlsx"
COMPOUNDS_PATH = r"/screened_compounds_rel_8.5.csv"

MOA_DIR = Path("/MoA")
MOA_FILES = {
    "clue_compoundinfo": MOA_DIR / "compoundinfo_beta.txt",
    "ctrp_compounds": MOA_DIR / "ctrp_v22_per_compound_clean.csv",
    "gdsc_compounds_84": MOA_DIR / "gdsc_screened_compounds_8_4_clean.csv",
    "prism_treatment_info": MOA_DIR / "primary-screen-replicate-collapsed-treatment-info.csv",
    "repurposing_hub": MOA_DIR / "repurposing_drugs_20200324.csv",
}

OUTPUT_DIR = Path("/Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def _norm(s: str) -> str:
    return (
        str(s).strip().lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace(".", "")
        .replace("(", "")
        .replace(")", "")
        .replace("\n", "")
    )

def find_col(df, candidates, required=True):
    norm_map = {_norm(c): c for c in df.columns}
    for cand in candidates:
        c = norm_map.get(_norm(cand))
        if c is not None:
            return c
    if required:
        raise KeyError(f"Could not find any of {candidates} in columns: {list(df.columns)[:80]}")
    return None

def read_table_auto(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    for sep in [",", "\t", ";"]:
        try:
            df = pd.read_csv(path, sep=sep, low_memory=False)
            if df.shape[1] > 1:
                return df
        except Exception:
            pass
    return pd.read_excel(path)

def normalize_drug_name(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    s = re.sub(r"[\(\)\[\]\{\}'\"`]", "", s)
    s = s.replace("_", " ").replace("-", " ").replace("/", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def split_multi(x):
    if pd.isna(x):
        return []
    s = str(x).replace("|", ",").replace(";", ",")
    return [p.strip() for p in s.split(",") if p and str(p).strip()]

def clean_gene_symbol_token(tok):
    tok = str(tok).strip().upper()
    tok = re.sub(r"\(.*?\)", "", tok).strip()
    return tok

def parse_target_genes(target_str):
    if pd.isna(target_str):
        return []
    toks = split_multi(target_str)
    out = []
    for t in toks:
        t = clean_gene_symbol_token(t)
        if re.fullmatch(r"[A-Z0-9\-]{2,15}", t):
            out.append(t)
    seen, uniq = set(), []
    for t in out:
        if t not in seen:
            seen.add(t)
            uniq.append(t)
    return uniq

def clean_expression_matrix(expr_raw: pd.DataFrame):
    expr = expr_raw.copy()

    gene_symbol_col = find_col(expr, ["GENE_SYMBOLS", "Gene Symbols", "gene_symbols"])
    gene_title_col = find_col(expr, ["GENE_title", "Gene title", "gene_title"], required=False)

    if gene_title_col is not None:
        expr = expr.drop(columns=[gene_title_col], errors="ignore")

    expr = expr.set_index(gene_symbol_col)
    expr.index = expr.index.astype(str).str.strip()
    expr = expr[expr.index.notna()]
    expr = expr[expr.index != ""]
    expr = expr[expr.index.str.lower() != "nan"]

    expr.columns = (
        pd.Index(expr.columns.astype(str))
        .str.replace(r"^DATA\.", "", regex=True)
        .str.replace(r"\.\d+$", "", regex=True)
        .str.strip()
    )

    numeric_mask = expr.columns.str.fullmatch(r"\d+")
    expr = expr.loc[:, numeric_mask]
    expr = expr.apply(pd.to_numeric, errors="coerce")

    if expr.columns.duplicated().any():
        expr = expr.T.groupby(level=0).mean().T

    if expr.index.duplicated().any():
        expr = expr.groupby(expr.index).mean()

    expr = expr.astype(np.float32)

    expr_t = expr.T.copy()
    expr_t.index = pd.to_numeric(expr_t.index, errors="coerce")
    expr_t = expr_t[~expr_t.index.isna()]
    expr_t.index = expr_t.index.astype(np.int64)
    expr_t.index.name = "COSMIC_ID"

    return expr, expr_t

def find_response_cols(df):
    cols = {}
    cols["COSMIC_ID"] = find_col(df, ["COSMIC_ID", "COSMIC ID", "COSMIC identifier"])
    cols["DRUG_ID"] = find_col(df, ["DRUG_ID", "DRUG ID", "drug_id"])
    cols["DRUG_NAME"] = find_col(df, ["DRUG_NAME", "DRUG NAME", "Drug name", "drug_name"], required=False)
    cols["CELL_LINE_NAME"] = find_col(df, ["CELL_LINE_NAME", "CELL LINE NAME", "Cell line name"], required=False)
    cols["LN_IC50"] = find_col(df, ["LN_IC50", "LN IC50", "logIC50", "log_IC50"], required=False)
    cols["AUC"] = find_col(df, ["AUC", "area_under_curve"], required=False)
    cols["RMSE"] = find_col(df, ["RMSE", "rmse"], required=False)
    cols["Z_SCORE"] = find_col(df, ["Z_SCORE", "Z SCORE", "z_score"], required=False)
    return cols

def find_compound_cols(df):
    cols = {}
    cols["drug_id"] = find_col(df, ["drug_id", "DRUG_ID", "Drug ID"])
    cols["drug_name"] = find_col(df, ["drug_name", "DRUG_NAME", "Drug name"], required=False)
    cols["targets"] = find_col(df, ["targets", "drug_target", "Drug target", "target"], required=False)
    cols["pathway_name"] = find_col(df, ["pathway_name", "Target Pathway", "target_pathway", "pathway"], required=False)
    cols["pubchem"] = find_col(df, ["pubchem", "pubchem_id", "pubchemid"], required=False)
    cols["synonyms"] = find_col(df, ["synonyms", "aliases", "alternative_names"], required=False)
    return cols

def find_cellline_cols(df):
    cols = {}
    cols["COSMIC_ID"] = find_col(df, ["COSMIC_ID", "COSMIC ID", "COSMIC identifier", "Cosmic ID"])
    cols["CELL_LINE_NAME"] = find_col(df, ["CELL_LINE_NAME", "Cell line name", "Sample Name"], required=False)
    cols["TISSUE"] = find_col(df, ["TCGA_DESC", "Tissue", "Tissue Type", "Site", "lineage"], required=False)
    cols["CANCER_TYPE"] = find_col(df, ["Histology", "Cancer Type", "cancer_type", "Histology subtype"], required=False)
    return cols


print("Loading core GDSC files...")
expr_raw = pd.read_csv(EXPR_PATH, sep="\t", low_memory=False)
cell_lines = pd.read_excel(CELL_LINES_PATH)
dose = pd.read_excel(DOSE_PATH)
compounds = pd.read_csv(COMPOUNDS_PATH, low_memory=False)

print("expr_raw:", expr_raw.shape)
print("cell_lines:", cell_lines.shape)
print("dose:", dose.shape)
print("compounds:", compounds.shape)

print("\nCleaning expression matrix...")
expr_genes_x_cells, expr_cells_x_genes = clean_expression_matrix(expr_raw)
print("expr_genes_x_cells:", expr_genes_x_cells.shape)
print("expr_cells_x_genes:", expr_cells_x_genes.shape)


print("\nCleaning dose response...")
dose_cols = find_response_cols(dose)
rename_map = {v: k for k, v in dose_cols.items() if v is not None}
dose_clean = dose.rename(columns=rename_map).copy()

keep_candidates = [
    "COSMIC_ID", "DRUG_ID", "DRUG_NAME", "CELL_LINE_NAME",
    "LN_IC50", "AUC", "RMSE", "Z_SCORE",
    "SANGER_MODEL_ID", "TCGA_DESC", "DATASET", "MAX_CONC", "MIN_CONC",
    "SCREENING_SITE", "SCREENING_MEDIUM", "CELL_ID"
]
keep_cols = [c for c in keep_candidates if c in dose_clean.columns]
dose_clean = dose_clean[keep_cols].copy()

dose_clean["COSMIC_ID"] = pd.to_numeric(dose_clean["COSMIC_ID"], errors="coerce")
dose_clean["DRUG_ID"] = pd.to_numeric(dose_clean["DRUG_ID"], errors="coerce")

for c in ["LN_IC50", "AUC", "RMSE", "Z_SCORE", "MAX_CONC", "MIN_CONC"]:
    if c in dose_clean.columns:
        dose_clean[c] = pd.to_numeric(dose_clean[c], errors="coerce")

dose_clean = dose_clean.dropna(subset=["COSMIC_ID", "DRUG_ID"])
dose_clean["COSMIC_ID"] = dose_clean["COSMIC_ID"].astype(np.int64)
dose_clean["DRUG_ID"] = dose_clean["DRUG_ID"].astype(np.int64)

print("dose_clean:", dose_clean.shape)

print("\nCleaning compounds annotation...")
compound_cols = find_compound_cols(compounds)
rename_cmp = {compound_cols["drug_id"]: "DRUG_ID"}
if compound_cols["drug_name"]:
    rename_cmp[compound_cols["drug_name"]] = "COMPOUND_NAME"
if compound_cols["targets"]:
    rename_cmp[compound_cols["targets"]] = "TARGET"
if compound_cols["pathway_name"]:
    rename_cmp[compound_cols["pathway_name"]] = "PATHWAY_NAME"
if compound_cols["pubchem"]:
    rename_cmp[compound_cols["pubchem"]] = "PUBCHEM"
if compound_cols["synonyms"]:
    rename_cmp[compound_cols["synonyms"]] = "SYNONYMS"

compounds_clean = compounds.rename(columns=rename_cmp).copy()
compounds_clean["DRUG_ID"] = pd.to_numeric(compounds_clean["DRUG_ID"], errors="coerce")
compounds_clean = compounds_clean.dropna(subset=["DRUG_ID"])
compounds_clean["DRUG_ID"] = compounds_clean["DRUG_ID"].astype(np.int64)
compounds_clean = compounds_clean.drop_duplicates(subset=["DRUG_ID"])

print("compounds_clean:", compounds_clean.shape)


print("\nCleaning cell line details...")
cell_cols = find_cellline_cols(cell_lines)
rename_cl = {cell_cols["COSMIC_ID"]: "COSMIC_ID"}
if cell_cols["CELL_LINE_NAME"]:
    rename_cl[cell_cols["CELL_LINE_NAME"]] = "CELL_LINE_NAME_ANN"
if cell_cols["TISSUE"]:
    rename_cl[cell_cols["TISSUE"]] = "TISSUE_ANN"
if cell_cols["CANCER_TYPE"]:
    rename_cl[cell_cols["CANCER_TYPE"]] = "CANCER_TYPE_ANN"

cell_lines_clean = cell_lines.rename(columns=rename_cl).copy()
cell_lines_clean["COSMIC_ID"] = pd.to_numeric(cell_lines_clean["COSMIC_ID"], errors="coerce")
cell_lines_clean = cell_lines_clean.dropna(subset=["COSMIC_ID"])
cell_lines_clean["COSMIC_ID"] = cell_lines_clean["COSMIC_ID"].astype(np.int64)
cell_lines_clean = cell_lines_clean.drop_duplicates(subset=["COSMIC_ID"])

print("cell_lines_clean:", cell_lines_clean.shape)

print("\nMerging dose + compounds + cell lines...")
resp_meta = (
    dose_clean
    .merge(compounds_clean, on="DRUG_ID", how="left")
    .merge(cell_lines_clean, on="COSMIC_ID", how="left")
)

print("resp_meta:", resp_meta.shape)

print("\nLoading MoA files...")
moa_raw = {}
for k, p in MOA_FILES.items():
    if p.exists():
        moa_raw[k] = read_table_auto(p)
        print(f"{k}: {p.name} -> {moa_raw[k].shape}")
    else:
        print(f"WARNING: Missing {p}")


print("\nStandardizing MoA sources...")
standardized_tables = []

if "clue_compoundinfo" in moa_raw:
    df = moa_raw["clue_compoundinfo"].copy()
    name_col = find_col(df, ["pert_iname", "compound_name", "name", "drug_name"], required=False)
    moa_col = find_col(df, ["moa", "moa_list", "mechanism_of_action"], required=False)
    target_col = find_col(df, ["target", "targets", "target_list", "gene_target", "protein_targets"], required=False)
    syn_col = find_col(df, ["alternative_names", "synonyms", "aliases"], required=False)
    id_col = find_col(df, ["pert_id", "broad_id", "compound_id", "cmap_name"], required=False)

    standardized_tables.append(pd.DataFrame({
        "source": "CLUE",
        "source_id": df[id_col] if id_col else pd.NA,
        "compound_name": df[name_col] if name_col else pd.NA,
        "synonyms_raw": df[syn_col] if syn_col else pd.NA,
        "moa_label": df[moa_col] if moa_col else pd.NA,
        "target_raw": df[target_col] if target_col else pd.NA,
        "pathway_name": pd.NA,
    }))

if "repurposing_hub" in moa_raw:
    df = moa_raw["repurposing_hub"].copy()
    name_col = find_col(df, ["pert_iname", "name", "compound_name", "drug_name"], required=False)
    moa_col = find_col(df, ["moa", "moa_list", "mechanism_of_action"], required=False)
    target_col = find_col(df, ["target", "targets", "target_list"], required=False)
    syn_col = find_col(df, ["aliases", "synonyms", "alternative_names"], required=False)
    id_col = find_col(df, ["broad_id", "pert_id", "compound_id", "pubchem_cid"], required=False)

    standardized_tables.append(pd.DataFrame({
        "source": "RepurposingHub",
        "source_id": df[id_col] if id_col else pd.NA,
        "compound_name": df[name_col] if name_col else pd.NA,
        "synonyms_raw": df[syn_col] if syn_col else pd.NA,
        "moa_label": df[moa_col] if moa_col else pd.NA,
        "target_raw": df[target_col] if target_col else pd.NA,
        "pathway_name": pd.NA,
    }))

if "prism_treatment_info" in moa_raw:
    df = moa_raw["prism_treatment_info"].copy()
    name_col = find_col(df, ["name", "compound_name", "pert_iname", "drug", "treatment"], required=False)
    moa_col = find_col(df, ["moa", "moa_list", "mechanism_of_action"], required=False)
    target_col = find_col(df, ["target", "targets", "target_list"], required=False)
    syn_col = find_col(df, ["synonyms", "aliases", "alternative_names"], required=False)
    id_col = find_col(df, ["broad_id", "pert_id", "compound_id"], required=False)

    standardized_tables.append(pd.DataFrame({
        "source": "PRISM",
        "source_id": df[id_col] if id_col else pd.NA,
        "compound_name": df[name_col] if name_col else pd.NA,
        "synonyms_raw": df[syn_col] if syn_col else pd.NA,
        "moa_label": df[moa_col] if moa_col else pd.NA,
        "target_raw": df[target_col] if target_col else pd.NA,
        "pathway_name": pd.NA,
    }))

if "gdsc_compounds_84" in moa_raw:
    df = moa_raw["gdsc_compounds_84"].copy()
    name_col = find_col(df, ["drug_name", "name", "compound_name"], required=False)
    target_col = find_col(df, ["target", "targets", "drug_target", "putative_target"], required=False)
    pathway_col = find_col(df, ["pathway_name", "target_pathway", "pathway"], required=False)
    syn_col = find_col(df, ["synonyms", "aliases", "alternative_names"], required=False)
    id_col = find_col(df, ["drug_id", "id", "compound_id"], required=False)

    standardized_tables.append(pd.DataFrame({
        "source": "GDSC8.4",
        "source_id": df[id_col] if id_col else pd.NA,
        "compound_name": df[name_col] if name_col else pd.NA,
        "synonyms_raw": df[syn_col] if syn_col else pd.NA,
        "moa_label": pd.NA,
        "target_raw": df[target_col] if target_col else pd.NA,
        "pathway_name": df[pathway_col] if pathway_col else pd.NA,
    }))

if "ctrp_compounds" in moa_raw:
    df = moa_raw["ctrp_compounds"].copy()
    name_col = find_col(df, ["cpd_name", "compound_name", "name", "drug_name"], required=False)
    moa_col = find_col(df, ["moa", "mechanism_of_action"], required=False)
    target_col = find_col(df, ["target", "targets", "gene_target", "protein_target"], required=False)
    pathway_col = find_col(df, ["pathway_name", "pathway", "gene_set"], required=False)
    syn_col = find_col(df, ["synonyms", "aliases", "cpd_name_alt"], required=False)
    id_col = find_col(df, ["master_cpd_id", "cpd_id", "compound_id"], required=False)

    standardized_tables.append(pd.DataFrame({
        "source": "CTRP",
        "source_id": df[id_col] if id_col else pd.NA,
        "compound_name": df[name_col] if name_col else pd.NA,
        "synonyms_raw": df[syn_col] if syn_col else pd.NA,
        "moa_label": df[moa_col] if moa_col else pd.NA,
        "target_raw": df[target_col] if target_col else pd.NA,
        "pathway_name": df[pathway_col] if pathway_col else pd.NA,
    }))

if len(standardized_tables) == 0:
    raise ValueError("No MoA tables were standardized. Check MoA file paths / columns.")

moa_unified_raw = pd.concat(standardized_tables, ignore_index=True)
moa_unified_raw["compound_name"] = moa_unified_raw["compound_name"].astype("string")
moa_unified_raw["compound_name_norm"] = moa_unified_raw["compound_name"].map(normalize_drug_name)
moa_unified_raw["moa_label"] = moa_unified_raw["moa_label"].astype("string")
moa_unified_raw["target_raw"] = moa_unified_raw["target_raw"].astype("string")
moa_unified_raw["pathway_name"] = moa_unified_raw["pathway_name"].astype("string")
moa_unified_raw["target_genes_list"] = moa_unified_raw["target_raw"].apply(parse_target_genes)
moa_unified_raw["target_genes"] = moa_unified_raw["target_genes_list"].apply(lambda x: ",".join(x) if len(x) else pd.NA)

print("moa_unified_raw:", moa_unified_raw.shape)


print("\nMapping MoA to GDSC compounds...")

gdsc_comp = compounds_clean.copy()
gdsc_comp["compound_name_gdsc"] = gdsc_comp["COMPOUND_NAME"].astype("string")
gdsc_comp["compound_name_norm"] = gdsc_comp["compound_name_gdsc"].map(normalize_drug_name)


drug_alias_map_raw = {


    # --- Navitoclax ---
    "ABT-263": "NAVITOCLAX",
    "ABT 263": "NAVITOCLAX",
    "ABT263": "NAVITOCLAX",
    "NAVITOCLAX (ABT-263)": "NAVITOCLAX",

    # --- JQ1 ---
    "(+)-JQ1": "JQ1",
    "(-)-JQ1": "JQ1",
    "JQ-1": "JQ1",

    # --- Vorinostat ---
    "SAHA": "VORINOSTAT",
    "ZOLINZA": "VORINOSTAT",

    # --- Vemurafenib ---
    "PLX4032": "VEMURAFENIB",

    # --- Oxaliplatin ---
    "ELOXATIN": "OXALIPLATIN",

    # --- Estramustine ---
    "ESTRAMUSTINEPHOSPHATESODIUM": "ESTRAMUSTINE",
    "EMCYT": "ESTRAMUSTINE",

    # --- Vinorelbine ---
    "NAVELBINEDITARTRATE": "VINORELBINE",
    "NAVELBINE": "VINORELBINE",

    # --- Etoposide ---
    "VEPESID": "ETOPOSIDE",
    "VEPESIDJ": "ETOPOSIDE",

    # --- Tamoxifen ---
    "TAMOXIFENCITRATE": "TAMOXIFEN",

    # --- Streptozocin ---
    "ZANOSAR": "STREPTOZOCIN",

    # --- Melphalan ---
    "MELPHALANHYDROCHLORIDE": "MELPHALAN",

    # --- Bendamustine ---
    "BENDAMUSTINEHYDROCHLORIDE": "BENDAMUSTINE",

    # --- Vincristine ---
    "VINCRISTINESULFATE": "VINCRISTINE",

    # --- Cytarabine ---
    "CYTARABINEHYDROCHLORIDE": "CYTARABINE",

    # --- Topotecan ---
    "TOPOTECANHYDROCHLORIDE": "TOPOTECAN",

    # --- Procarbazine ---
    "PROCARBAZINEHYDROCHLORIDE": "PROCARBAZINE",

    # --- Quinacrine ---
    "QUINACRINEHYDROCHLORIDE": "QUINACRINE",

    # --- Fludarabine ---
    "FLUDARABINEBASE": "FLUDARABINE",

    # --- Pazopanib ---
    "PAZOPANIBHYDROCHLORIDE": "PAZOPANIB",

    # --- Erlotinib ---
    "ERLOTINIBHYDROCHLORIDE": "ERLOTINIB",

    # --- Mechlorethamine ---
    "MECHLORETHAMINEHYDROCHLORIDE": "MECHLORETHAMINE",

    # --- Gemcitabine ---
    "GEMZAR": "GEMCITABINE",

    # --- Azacitidine ---
    "5AZACYTIDINE": "AZACITIDINE",
    "AZACYTIDINE5": "AZACITIDINE",
    "AZACYTIDINE": "AZACITIDINE",

    # --- Carboplatin ---
    "CARBOPLATINUM": "CARBOPLATIN",

    # --- Fluorouracil family ---
    "5FU": "FLUOROURACIL",
    "5-FLUOROURACIL": "FLUOROURACIL",
    "5FLUORO2DEOXYURIDINE": "FLOXURIDINE",

    # --- Aminolevulinic acid ---
    "AMINOLEVULINICACIDHYDROCHLORIDE": "AMINOLEVULINICACID",
    "5AMINOLEVULINICACIDHYDROCHLORIDE": "AMINOLEVULINICACID",

    # --- Special cases ---
    "DIARSENICTRIOXIDE": "ARSENICTRIOXIDE",
    "TRISENOX": "ARSENICTRIOXIDE",
    "TRIETHYLENEMELAMINE": "TRETAMINE",
    "DIAMMINEDICHLOROPLATINUM": "CISPLATIN",
}

drug_alias_map = {
    normalize_drug_name(k): normalize_drug_name(v)
    for k, v in drug_alias_map_raw.items()
}

def canonicalize_drug_name(x):
    if pd.isna(x):
        return pd.NA
    nx = normalize_drug_name(x)
    return drug_alias_map.get(nx, nx)

gdsc_comp["compound_name_canonical"] = gdsc_comp["compound_name_gdsc"].map(canonicalize_drug_name)

lookup_rows = []
for _, r in moa_unified_raw.iterrows():
    candidate_names = []

    if pd.notna(r.get("compound_name", pd.NA)):
        candidate_names.append(r["compound_name"])

    if pd.notna(r.get("synonyms_raw", pd.NA)):
        candidate_names.extend(split_multi(r["synonyms_raw"]))

    for raw_nm in candidate_names:
        norm_nm = normalize_drug_name(raw_nm)
        canon_nm = canonicalize_drug_name(raw_nm)

        if pd.notna(norm_nm):
            lookup_rows.append({
                "match_name_key": norm_nm,
                "match_key_type": "normalized",
                "source": r["source"],
                "source_id": r["source_id"],
                "compound_name_src": r["compound_name"],
                "moa_label": r["moa_label"],
                "target_raw": r["target_raw"],
                "target_genes": r["target_genes"],
                "pathway_name_moa": r["pathway_name"],
            })

        if pd.notna(canon_nm):
            lookup_rows.append({
                "match_name_key": canon_nm,
                "match_key_type": "canonical",
                "source": r["source"],
                "source_id": r["source_id"],
                "compound_name_src": r["compound_name"],
                "moa_label": r["moa_label"],
                "target_raw": r["target_raw"],
                "target_genes": r["target_genes"],
                "pathway_name_moa": r["pathway_name"],
            })

moa_lookup = pd.DataFrame(lookup_rows).drop_duplicates()

gdsc_match_canon = gdsc_comp.merge(
    moa_lookup,
    left_on="compound_name_canonical",
    right_on="match_name_key",
    how="left"
)

unmatched_drugs = gdsc_match_canon.loc[gdsc_match_canon["source"].isna(), "DRUG_ID"].dropna().unique()
gdsc_unmatched = gdsc_comp[gdsc_comp["DRUG_ID"].isin(unmatched_drugs)].copy()

gdsc_match_norm = gdsc_unmatched.merge(
    moa_lookup,
    left_on="compound_name_norm",
    right_on="match_name_key",
    how="left"
)

gdsc_moa_matches = pd.concat([
    gdsc_match_canon[gdsc_match_canon["source"].notna()],
    gdsc_match_norm[gdsc_match_norm["source"].notna()]
], ignore_index=True)

matched_ids = set(gdsc_moa_matches["DRUG_ID"].dropna().astype(int).tolist()) if not gdsc_moa_matches.empty else set()
gdsc_still_unmatched = gdsc_comp[~gdsc_comp["DRUG_ID"].isin(matched_ids)].copy()

for c in ["source","source_id","compound_name_src","moa_label","target_raw","target_genes","pathway_name_moa","match_name_key","match_key_type"]:
    gdsc_still_unmatched[c] = pd.NA

gdsc_moa_matches = pd.concat([gdsc_moa_matches, gdsc_still_unmatched], ignore_index=True, sort=False)

source_priority = {"CLUE": 5, "RepurposingHub": 4, "PRISM": 3, "GDSC8.4": 2, "CTRP": 1}
match_priority = {"canonical": 2, "normalized": 1}

gdsc_moa_matches["source_priority"] = gdsc_moa_matches["source"].map(source_priority).fillna(0)
gdsc_moa_matches["match_priority"] = gdsc_moa_matches["match_key_type"].map(match_priority).fillna(0)
gdsc_moa_matches["has_target_genes"] = gdsc_moa_matches["target_genes"].notna().astype(int)
gdsc_moa_matches["has_moa"] = gdsc_moa_matches["moa_label"].notna().astype(int)

gdsc_moa_best = (
    gdsc_moa_matches
    .sort_values(
        ["DRUG_ID", "has_target_genes", "has_moa", "match_priority", "source_priority"],
        ascending=[True, False, False, False, False]
    )
    .drop_duplicates(subset=["DRUG_ID"], keep="first")
    .copy()
)

gdsc_moa_best["TARGET_GENES_FINAL"] = gdsc_moa_best["target_genes"]

if "TARGET" in gdsc_moa_best.columns:
    miss = gdsc_moa_best["TARGET_GENES_FINAL"].isna()
    gdsc_moa_best.loc[miss, "TARGET_GENES_FINAL"] = gdsc_moa_best.loc[miss, "TARGET"].apply(
        lambda x: ",".join(parse_target_genes(x)) if pd.notna(x) and len(parse_target_genes(x)) > 0 else pd.NA
    )

gdsc_moa_best["MOA_LABEL_FINAL"] = gdsc_moa_best["moa_label"]
gdsc_moa_best["PATHWAY_NAME_FINAL"] = gdsc_moa_best["pathway_name_moa"]

if "PATHWAY_NAME" in gdsc_moa_best.columns:
    gdsc_moa_best["PATHWAY_NAME_FINAL"] = gdsc_moa_best["PATHWAY_NAME_FINAL"].fillna(gdsc_moa_best["PATHWAY_NAME"])

gdsc_moa_best["MOA_SOURCE_FINAL"] = gdsc_moa_best["source"].fillna("GDSC_only")

gdsc_moa_final = gdsc_moa_best[[
    "DRUG_ID", "MOA_LABEL_FINAL", "TARGET_GENES_FINAL", "PATHWAY_NAME_FINAL", "MOA_SOURCE_FINAL"
]].drop_duplicates(subset=["DRUG_ID"]).copy()

print("gdsc_moa_final:", gdsc_moa_final.shape)


print("\nMerging MoA into response metadata...")
resp_meta_moa = resp_meta.merge(gdsc_moa_final, on="DRUG_ID", how="left")
print("resp_meta_moa:", resp_meta_moa.shape)

print("\nBuilding all_merged_full...")
all_merged_full = resp_meta_moa.merge(
    expr_cells_x_genes,
    left_on="COSMIC_ID",
    right_index=True,
    how="inner"
)

if "LN_IC50" in all_merged_full.columns:
    all_merged_full = all_merged_full.dropna(subset=["LN_IC50"]).copy()

all_merged_full.columns = [str(c).replace("\n", " ").strip() for c in all_merged_full.columns]

print("all_merged_full:", all_merged_full.shape)


print("\nSaving all_merged_full...")


save_df = all_merged_full.copy()


obj_cols = save_df.select_dtypes(include=["object"]).columns
for c in obj_cols:
    save_df[c] = save_df[c].astype("string")


out_parquet = OUTPUT_DIR / "all_merged_full_resp_meta_moa_expression.parquet"
save_df.to_parquet(out_parquet, index=False)


preview_cols = save_df.columns[:200]
out_preview_xlsx = OUTPUT_DIR / "all_merged_full_preview.xlsx"
with pd.ExcelWriter(out_preview_xlsx, engine="openpyxl") as writer:
    save_df.loc[:, preview_cols].head(1000).to_excel(
        writer, sheet_name="preview", index=False
    )

print("Saved:")
print(" -", out_parquet)
print(" -", out_preview_xlsx)

print("\nPreview:")
display(save_df.iloc[:5, :10])
display(all_merged_full.iloc[:5, :10])

In [ ]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict

print("\nLoading STRING PPI files...")


ppi_folder = "/Datasets/PPI/"
links_path = os.path.join(ppi_folder, "9606.protein.links.v12.0.txt")
info_path  = os.path.join(ppi_folder, "9606.protein.info.v12.0.txt")

STRING_SCORE_THRESHOLD = 700 


if not os.path.exists(links_path):
    raise FileNotFoundError(f"STRING links file not found: {links_path}")
if not os.path.exists(info_path):
    raise FileNotFoundError(f"STRING info file not found: {info_path}")


ppi_info = pd.read_csv(info_path, sep="\t", low_memory=False)


ppi_links = pd.read_csv(links_path, sep=r"\s+", engine="python")

print("ppi_info:", ppi_info.shape)
print("ppi_links (raw):", ppi_links.shape)


def _norm_local(s: str) -> str:
    return (
        str(s).strip().lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace(".", "")
    )

def find_col_local(df, candidates, required=True):
    norm_map = {_norm_local(c): c for c in df.columns}
    for cand in candidates:
        c = norm_map.get(_norm_local(cand))
        if c is not None:
            return c
    if required:
        raise KeyError(f"Could not find any of {candidates} in columns: {list(df.columns)}")
    return None


col_protein_ext = find_col_local(ppi_info, ["#string_protein_id", "string_protein_id", "protein_external_id", "protein_externalid", "protein"])
col_preferred   = find_col_local(ppi_info, ["preferred_name", "preferredname", "gene", "gene_name"], required=False)

if col_preferred is None:
    raise ValueError("Could not find preferred_name (gene symbol) column in STRING info file.")


col_p1    = find_col_local(ppi_links, ["protein1"])
col_p2    = find_col_local(ppi_links, ["protein2"])
col_score = find_col_local(ppi_links, ["combined_score", "combinedscore", "score"])


ppi_info_clean = ppi_info[[col_protein_ext, col_preferred]].copy()
ppi_info_clean.columns = ["protein_id", "gene_symbol"]
ppi_info_clean["protein_id"] = ppi_info_clean["protein_id"].astype(str).str.strip()
ppi_info_clean["gene_symbol"] = ppi_info_clean["gene_symbol"].astype(str).str.strip()


ppi_info_clean["gene_symbol_u"] = ppi_info_clean["gene_symbol"].str.upper()

protein_to_gene = dict(zip(ppi_info_clean["protein_id"], ppi_info_clean["gene_symbol_u"]))


ppi_edges = ppi_links[[col_p1, col_p2, col_score]].copy()
ppi_edges.columns = ["protein1", "protein2", "combined_score"]

ppi_edges["combined_score"] = pd.to_numeric(ppi_edges["combined_score"], errors="coerce")
ppi_edges = ppi_edges.dropna(subset=["combined_score"])
ppi_edges = ppi_edges[ppi_edges["combined_score"] > STRING_SCORE_THRESHOLD].copy()


ppi_edges["gene1"] = ppi_edges["protein1"].map(protein_to_gene)
ppi_edges["gene2"] = ppi_edges["protein2"].map(protein_to_gene)


ppi_edges = ppi_edges.dropna(subset=["gene1", "gene2"]).copy()
ppi_edges = ppi_edges[ppi_edges["gene1"] != ppi_edges["gene2"]].copy()


g1 = ppi_edges["gene1"].astype(str).values
g2 = ppi_edges["gene2"].astype(str).values
swap_mask = g1 > g2

gene_a = np.where(swap_mask, g2, g1)
gene_b = np.where(swap_mask, g1, g2)

ppi_edges["gene_a"] = gene_a
ppi_edges["gene_b"] = gene_b


ppi_edges = (
    ppi_edges.groupby(["gene_a", "gene_b"], as_index=False)["combined_score"]
    .max()
    .rename(columns={"combined_score": "string_score"})
)

print("ppi_edges (high-confidence, gene-level):", ppi_edges.shape)
print("Unique genes in PPI:", pd.unique(pd.concat([ppi_edges["gene_a"], ppi_edges["gene_b"]])).shape[0])


neighbors_by_gene = defaultdict(set)
neighbor_scores_by_gene = defaultdict(dict)

for _, r in ppi_edges.iterrows():
    a = r["gene_a"]
    b = r["gene_b"]
    s = float(r["string_score"])
    neighbors_by_gene[a].add(b)
    neighbors_by_gene[b].add(a)


    
    prev_ab = neighbor_scores_by_gene[a].get(b, -np.inf)
    prev_ba = neighbor_scores_by_gene[b].get(a, -np.inf)
    if s > prev_ab:
        neighbor_scores_by_gene[a][b] = s
    if s > prev_ba:
        neighbor_scores_by_gene[b][a] = s

ppi_degree = {g: len(nbrs) for g, nbrs in neighbors_by_gene.items()}

print("Adjacency built for genes:", len(neighbors_by_gene))


expr_gene_cols_full = [g for g in expr_cells_x_genes.columns if g in all_merged_full.columns]
gene_upper_to_col_full = {str(g).upper(): g for g in expr_gene_cols_full}

print("Expression gene columns in all_merged_full:", len(expr_gene_cols_full))


def parse_targets_for_ppi(s):
    """
    Uses your existing parse_target_genes() from earlier code.
    Returns uppercase target genes that exist in expression gene universe.
    """
    if pd.isna(s):
        return []
    tgts = parse_target_genes(s)  
    tgts = [str(t).upper() for t in tgts if pd.notna(t)]

    seen = set()
    out = []
    for t in tgts:
        if t not in seen:
            seen.add(t)
            out.append(t)
    return out


drug_target_base_cols = [c for c in [
    "DRUG_ID", "COMPOUND_NAME", "DRUG_NAME", "TARGET_GENES_FINAL", "MOA_LABEL_FINAL",
    "PATHWAY_NAME_FINAL", "MOA_SOURCE_FINAL"
] if c in all_merged_full.columns]

drug_target_base = (
    all_merged_full[drug_target_base_cols]
    .drop_duplicates(subset=["DRUG_ID"])
    .copy()
)

drug_target_base["target_genes_ppi_raw"] = drug_target_base["TARGET_GENES_FINAL"].apply(parse_targets_for_ppi)


drug_target_base["target_genes_in_expr"] = drug_target_base["target_genes_ppi_raw"].apply(
    lambda xs: [g for g in xs if g in gene_upper_to_col_full]
)


drug_target_base["target_genes_in_ppi"] = drug_target_base["target_genes_in_expr"].apply(
    lambda xs: [g for g in xs if g in neighbors_by_gene]
)

def get_union_neighbors(genes):
    nbrs = set()
    for g in genes:
        nbrs |= neighbors_by_gene.get(g, set())
    return sorted(nbrs)

drug_target_base["ppi_neighbor_genes_all"] = drug_target_base["target_genes_in_ppi"].apply(get_union_neighbors)


drug_target_base["ppi_neighbor_genes_in_expr"] = drug_target_base.apply(
    lambda r: [
        g for g in r["ppi_neighbor_genes_all"]
        if (g not in set(r["target_genes_in_expr"])) and (g in gene_upper_to_col_full)
    ],
    axis=1
)


drug_target_base["ppi_num_targets_total"] = drug_target_base["target_genes_ppi_raw"].apply(len)
drug_target_base["ppi_num_targets_in_expr"] = drug_target_base["target_genes_in_expr"].apply(len)
drug_target_base["ppi_num_targets_in_ppi"] = drug_target_base["target_genes_in_ppi"].apply(len)
drug_target_base["ppi_num_neighbors_in_expr"] = drug_target_base["ppi_neighbor_genes_in_expr"].apply(len)


def degree_stats(genes):
    if not genes:
        return pd.Series({
            "ppi_target_degree_mean": np.nan,
            "ppi_target_degree_max": np.nan,
            "ppi_target_degree_min": np.nan
        })
    degs = [ppi_degree.get(g, 0) for g in genes]
    return pd.Series({
        "ppi_target_degree_mean": float(np.mean(degs)),
        "ppi_target_degree_max": float(np.max(degs)),
        "ppi_target_degree_min": float(np.min(degs))
    })

drug_target_base = pd.concat(
    [drug_target_base, drug_target_base["target_genes_in_ppi"].apply(degree_stats)],
    axis=1
)


drug_target_base["target_genes_in_expr_str"] = drug_target_base["target_genes_in_expr"].apply(
    lambda xs: ",".join(xs) if len(xs) else pd.NA
)
drug_target_base["ppi_neighbor_genes_in_expr_str"] = drug_target_base["ppi_neighbor_genes_in_expr"].apply(
    lambda xs: ",".join(xs) if len(xs) else pd.NA
)

ppi_static_cols = [c for c in [
    "DRUG_ID",
    "ppi_num_targets_total",
    "ppi_num_targets_in_expr",
    "ppi_num_targets_in_ppi",
    "ppi_num_neighbors_in_expr",
    "ppi_target_degree_mean",
    "ppi_target_degree_max",
    "ppi_target_degree_min",
    "target_genes_in_expr_str",
    "ppi_neighbor_genes_in_expr_str",
] if c in drug_target_base.columns]

for c in ppi_static_cols:
    if c != "DRUG_ID" and c in all_merged_full.columns:
        all_merged_full = all_merged_full.drop(columns=[c])

all_merged_full = all_merged_full.merge(
    drug_target_base[ppi_static_cols].drop_duplicates(subset=["DRUG_ID"]),
    on="DRUG_ID",
    how="left"
)

print("all_merged_full after static PPI merge:", all_merged_full.shape)


print("\nComputing row-level PPI expression features on all_merged_full...")


for c in [
    "ppi_neighbor_expr_mean",
    "ppi_neighbor_expr_max",
    "ppi_neighbor_expr_min",
    "ppi_neighbor_expr_std",
    "ppi_target_minus_neighbor_expr_mean", 
]:
    all_merged_full[c] = np.nan


drug_to_neighbor_cols = {}
for _, r in drug_target_base.iterrows():
    did = r["DRUG_ID"]
    nbrs = r["ppi_neighbor_genes_in_expr"]
    cols = [gene_upper_to_col_full[g] for g in nbrs if g in gene_upper_to_col_full]

    cols = list(dict.fromkeys(cols))
    drug_to_neighbor_cols[did] = cols


for did, cols in drug_to_neighbor_cols.items():
    if len(cols) == 0:
        continue

    idx = all_merged_full["DRUG_ID"] == did
    if idx.sum() == 0:
        continue

    Xn = all_merged_full.loc[idx, cols].apply(pd.to_numeric, errors="coerce")

    all_merged_full.loc[idx, "ppi_neighbor_expr_mean"] = Xn.mean(axis=1, skipna=True).values
    all_merged_full.loc[idx, "ppi_neighbor_expr_max"]  = Xn.max(axis=1, skipna=True).values
    all_merged_full.loc[idx, "ppi_neighbor_expr_min"]  = Xn.min(axis=1, skipna=True).values
    all_merged_full.loc[idx, "ppi_neighbor_expr_std"]  = Xn.std(axis=1, skipna=True).values

if "moa_target_expr_mean" in all_merged_full.columns:
    all_merged_full["ppi_target_minus_neighbor_expr_mean"] = (
        pd.to_numeric(all_merged_full["moa_target_expr_mean"], errors="coerce")
        - pd.to_numeric(all_merged_full["ppi_neighbor_expr_mean"], errors="coerce")
    )


print("\n================ PPI INTEGRATION SUMMARY ================")
print("STRING threshold:", STRING_SCORE_THRESHOLD, "(kept > threshold)")
print("High-confidence PPI edges (gene-level):", ppi_edges.shape[0])
print("Drugs in all_merged_full:", all_merged_full["DRUG_ID"].nunique())
print("Rows in all_merged_full:", all_merged_full.shape[0])

ppi_cols_added = [
    "ppi_num_targets_total",
    "ppi_num_targets_in_expr",
    "ppi_num_targets_in_ppi",
    "ppi_num_neighbors_in_expr",
    "ppi_target_degree_mean",
    "ppi_target_degree_max",
    "ppi_target_degree_min",
    "ppi_neighbor_expr_mean",
    "ppi_neighbor_expr_max",
    "ppi_neighbor_expr_min",
    "ppi_neighbor_expr_std",
    "ppi_target_minus_neighbor_expr_mean",
    "target_genes_in_expr_str",
    "ppi_neighbor_genes_in_expr_str",
]
ppi_cols_added = [c for c in ppi_cols_added if c in all_merged_full.columns]

print("PPI columns added to all_merged_full:")
print(ppi_cols_added)

print("\nPPI coverage:")
for c in ["ppi_num_targets_in_ppi", "ppi_num_neighbors_in_expr", "ppi_neighbor_expr_mean"]:
    if c in all_merged_full.columns:
        print(f"  {c} non-null fraction: {all_merged_full[c].notna().mean():.4f}")


peek_cols = [c for c in [
    "DRUG_ID", "COMPOUND_NAME", "TARGET_GENES_FINAL", "MOA_LABEL_FINAL",
    "ppi_num_targets_in_ppi", "ppi_num_neighbors_in_expr",
    "ppi_target_degree_mean", "ppi_neighbor_expr_mean", "ppi_neighbor_expr_max"
] if c in all_merged_full.columns]
display(all_merged_full[peek_cols].head())



In [ ]:
import re
import time
import requests
import pandas as pd
from urllib.parse import quote
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

PUBCHEM_BASE = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"
MAX_WORKERS = 6
REQUEST_TIMEOUT = 20
RETRY_SLEEP = 1.5
MAX_RETRIES = 2

session = requests.Session()
session.headers.update({"User-Agent": "drug-structure-enrichment/2.0"})



def clean_str(x):
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    return x

def norm_key(x):
    return re.sub(r"[^a-z0-9]+", "", clean_str(x).lower())

def split_synonyms(x):
    x = clean_str(x)
    if not x:
        return []
    parts = re.split(r"\s*[;,|]\s*", x)
    return [p for p in parts if clean_str(p)]

def first_nonempty(series):
    for v in series:
        if clean_str(v):
            return clean_str(v)
    return pd.NA

def first_present(*vals):
    for v in vals:
        if clean_str(v):
            return v
    return None

def source_of_first_present(options):
    """
    options = [("PubChem", val1), ("ChEMBL", val2), ...]
    returns source name for first non-empty value
    """
    for src, val in options:
        if clean_str(val):
            return src
    return None

def make_candidate_names(row):
    candidates = []

    for col in ["DRUG_NAME", "COMPOUND_NAME"]:
        if col in row and clean_str(row[col]):
            candidates.append(clean_str(row[col]))

    if "SYNONYMS" in row and clean_str(row["SYNONYMS"]):
        candidates.extend(split_synonyms(row["SYNONYMS"]))

    extra = []
    for c in candidates:
        c2 = re.sub(r"\s*\(.*?\)\s*", " ", c).strip()
        c2 = re.sub(r"\s+", " ", c2).strip()
        if c2 and c2 != c:
            extra.append(c2)

    candidates.extend(extra)

    bad = {"", "na", "n/a", "<na>", "none", "nan"}
    seen = set()
    out = []
    for c in candidates:
        c = clean_str(c)
        if c.lower() in bad:
            continue
        k = norm_key(c)
        if k and k not in seen:
            seen.add(k)
            out.append(c)
    return out

def safe_get_json(url, params=None):
    for attempt in range(MAX_RETRIES + 1):
        try:
            r = session.get(url, params=params, timeout=REQUEST_TIMEOUT)
            if r.status_code == 404:
                return None
            r.raise_for_status()
            return r.json()
        except Exception:
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP * (attempt + 1))
            else:
                return None



def pubchem_lookup_one_name(name):
    url = (
        f"{PUBCHEM_BASE}/compound/name/"
        f"{quote(name, safe='')}/property/"
        "CanonicalSMILES,IsomericSMILES,InChI,InChIKey/JSON"
    )
    data = safe_get_json(url)
    if not data:
        return None

    props = data.get("PropertyTable", {}).get("Properties", [])
    if not props:
        return None

    p = props[0]
    return {
        "pubchem_match_name": name,
        "pubchem_cid": p.get("CID"),
        "pubchem_canonical_smiles": p.get("CanonicalSMILES"),
        "pubchem_isomeric_smiles": p.get("IsomericSMILES"),
        "pubchem_inchi": p.get("InChI"),
        "pubchem_inchikey": p.get("InChIKey"),
    }

def score_chembl_result(mol, query_name):
    score = 0
    qn = norm_key(query_name)
    pref = clean_str(mol.get("pref_name", ""))
    pref_n = norm_key(pref)

    if pref_n == qn:
        score += 100
    elif pref and pref.lower() == query_name.lower():
        score += 80

    if mol.get("molecule_structures"):
        score += 10

    if mol.get("max_phase") not in [None, ""]:
        try:
            score += float(mol.get("max_phase", 0))
        except Exception:
            pass

    return score

def chembl_lookup_one_name(name):
    url = f"{CHEMBL_BASE}/molecule/search"
    data = safe_get_json(url, params={"q": name, "format": "json"})
    if not data:
        return None

    molecules = data.get("molecules", [])
    if not molecules:
        return None

    best = sorted(
        molecules,
        key=lambda m: score_chembl_result(m, name),
        reverse=True
    )[0]

    structs = best.get("molecule_structures") or {}

    return {
        "chembl_match_name": name,
        "chembl_id": best.get("molecule_chembl_id"),
        "chembl_pref_name": best.get("pref_name"),
        "chembl_smiles": structs.get("canonical_smiles"),
        "chembl_inchi": structs.get("standard_inchi"),
        "chembl_inchikey": structs.get("standard_inchi_key"),
    }



def resolve_best_structure(pubchem_hit, chembl_hit):
    pubchem_hit = pubchem_hit or {}
    chembl_hit = chembl_hit or {}

    pubchem_smiles_best = first_present(
        pubchem_hit.get("pubchem_isomeric_smiles"),
        pubchem_hit.get("pubchem_canonical_smiles"),
    )
    chembl_smiles_best = first_present(
        chembl_hit.get("chembl_smiles"),
    )

    pubchem_inchi = first_present(pubchem_hit.get("pubchem_inchi"))
    chembl_inchi = first_present(chembl_hit.get("chembl_inchi"))

    pubchem_inchikey = first_present(pubchem_hit.get("pubchem_inchikey"))
    chembl_inchikey = first_present(chembl_hit.get("chembl_inchikey"))


    smiles_best = first_present(pubchem_smiles_best, chembl_smiles_best)
    inchi_best = first_present(pubchem_inchi, chembl_inchi)
    inchikey_best = first_present(pubchem_inchikey, chembl_inchikey)

    smiles_source_best = source_of_first_present([
        ("PubChem", pubchem_smiles_best),
        ("ChEMBL", chembl_smiles_best),
    ])
    inchi_source_best = source_of_first_present([
        ("PubChem", pubchem_inchi),
        ("ChEMBL", chembl_inchi),
    ])
    inchikey_source_best = source_of_first_present([
        ("PubChem", pubchem_inchikey),
        ("ChEMBL", chembl_inchikey),
    ])


    picked_sources = [s for s in [smiles_source_best, inchi_source_best, inchikey_source_best] if s]
    if len(set(picked_sources)) == 0:
        source_best = None
    elif len(set(picked_sources)) == 1:
        source_best = picked_sources[0]
    else:
        source_best = "Mixed"

    pub_inchikey = clean_str(pubchem_inchikey)
    ch_inchikey = clean_str(chembl_inchikey)

    id_disagreement = bool(pub_inchikey and ch_inchikey and pub_inchikey != ch_inchikey)

    return {
        "smiles_best": smiles_best,
        "inchi_best": inchi_best,
        "inchikey_best": inchikey_best,
        "smiles_source_best": smiles_source_best,
        "inchi_source_best": inchi_source_best,
        "inchikey_source_best": inchikey_source_best,
        "source_best": source_best,
        "pubchem_chembl_inchikey_disagree": id_disagreement,
    }

def resolve_drug(row):
    row = dict(row)
    candidates = make_candidate_names(row)

    pubchem_hit = None
    chembl_hit = None

    for name in candidates:
        if pubchem_hit is None:
            pubchem_hit = pubchem_lookup_one_name(name)
        if chembl_hit is None:
            chembl_hit = chembl_lookup_one_name(name)
        if pubchem_hit is not None and chembl_hit is not None:
            break

    pubchem_hit = pubchem_hit or {}
    chembl_hit = chembl_hit or {}
    best_struct = resolve_best_structure(pubchem_hit, chembl_hit)

    return {
        "DRUG_ID": row.get("DRUG_ID"),
        **pubchem_hit,
        **chembl_hit,
        **best_struct,
        "lookup_candidates_used": " | ".join(candidates[:10]),
        "n_lookup_candidates": len(candidates),
    }



if "DRUG_ID" not in all_merged_full.columns:
    raise ValueError("all_merged_full must contain DRUG_ID")

name_cols_available = [c for c in ["DRUG_NAME", "COMPOUND_NAME", "SYNONYMS"] if c in all_merged_full.columns]


agg_dict = {c: first_nonempty for c in name_cols_available}
drug_query_df = (
    all_merged_full[["DRUG_ID"] + name_cols_available]
    .groupby("DRUG_ID", as_index=False)
    .agg(agg_dict)
    .copy()
)

print("Unique drugs for lookup:", drug_query_df.shape[0])

results = []
records = drug_query_df.to_dict(orient="records")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(resolve_drug, row): i for i, row in enumerate(records)}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Fetching PubChem + ChEMBL"):
        try:
            results.append(fut.result())
        except Exception as e:
            idx = futures[fut]
            results.append({
                "DRUG_ID": drug_query_df.iloc[idx]["DRUG_ID"],
                "lookup_error": str(e)
            })

drug_lookup_df = pd.DataFrame(results)


cols_to_replace = [
    "pubchem_match_name", "pubchem_cid", "pubchem_canonical_smiles", "pubchem_isomeric_smiles",
    "pubchem_inchi", "pubchem_inchikey",
    "chembl_match_name", "chembl_id", "chembl_pref_name", "chembl_smiles", "chembl_inchi", "chembl_inchikey",
    "smiles_best", "inchi_best", "inchikey_best",
    "smiles_source_best", "inchi_source_best", "inchikey_source_best", "source_best",
    "pubchem_chembl_inchikey_disagree",
    "lookup_candidates_used", "n_lookup_candidates", "lookup_error"
]

existing_replace_cols = [c for c in cols_to_replace if c in all_merged_full.columns]
if existing_replace_cols:
    all_merged_full = all_merged_full.drop(columns=existing_replace_cols)

lookup_only_cols = [c for c in drug_lookup_df.columns if c != "DRUG_ID"]

all_merged_full_enriched = all_merged_full.merge(
    drug_lookup_df[["DRUG_ID"] + lookup_only_cols],
    on="DRUG_ID",
    how="left"
)

review_mismatch = drug_lookup_df.loc[
    drug_lookup_df["pubchem_chembl_inchikey_disagree"].fillna(False)
].copy()

print("Done.")
print("drug_query_df shape           :", drug_query_df.shape)
print("drug_lookup_df shape          :", drug_lookup_df.shape)
print("all_merged_full_enriched shape:", all_merged_full_enriched.shape)
print()

print("Coverage summary:")
summary_cols = [
    "pubchem_canonical_smiles",
    "pubchem_inchi",
    "chembl_smiles",
    "chembl_inchi",
    "smiles_best",
    "inchi_best",
    "inchikey_best",
]
for c in summary_cols:
    if c in drug_lookup_df.columns:
        print(f"{c:28s} {drug_lookup_df[c].notna().sum():4d} / {len(drug_lookup_df)}")

print()
print("Rows with PubChem/ChEMBL InChIKey disagreement:", len(review_mismatch))

display_cols = [c for c in [
    "DRUG_ID", "DRUG_NAME", "COMPOUND_NAME", "SYNONYMS",
    "pubchem_match_name", "pubchem_cid", "pubchem_canonical_smiles", "pubchem_isomeric_smiles",
    "pubchem_inchi", "pubchem_inchikey",
    "chembl_match_name", "chembl_id", "chembl_pref_name",
    "chembl_smiles", "chembl_inchi", "chembl_inchikey",
    "smiles_best", "inchi_best", "inchikey_best",
    "smiles_source_best", "inchi_source_best", "inchikey_source_best",
    "source_best", "pubchem_chembl_inchikey_disagree"
] if c in all_merged_full_enriched.columns]

print("\nPreview:")
display(all_merged_full_enriched[display_cols].head())


out_pkl = OUTPUT_DIR / "all_merged_full_enriched_with_structures_v3.pkl"
all_merged_full_enriched.to_pickle(out_pkl)
print("Saved:", out_pkl)